<a href="https://colab.research.google.com/github/Arzoo34/GEN-AI-with-Python/blob/main/BasicAgents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THE SIMPLE REFLEX AGENT


In [1]:
WORLD = {
    'A' : ['B', 'C'],
    'B' : ['A', 'D'],
    'C' : ['A', 'D', 'E'],
    'D' : ['B', 'C', 'E', 'F'],
    'E' : ['C', 'D', 'F'],
    'F' : ['D', 'E']
}
START_ROOM = 'A'
END_ROOM = 'F'

In [4]:
class ReflexAgent:
  def __init__(self):
    self.current_room = START_ROOM
    print("Reflex agent starts in {self.current_room}.")
  def percieve(self,world):
    # agent only sees current and next neighbour
    return self.current_room, world[self.current_room]
  def think(self,perception):
    current,neighbors = perception
    if current == END_ROOM:
      return "CELEBRATE"
      # go to first neighbor alphabetically
    return f"MOVE_TO_{neighbors[0]}"
  def act(self,action):
    if action.startswith("MOVE_TO_"):
      new_room = action.split("_")[-1]
      self.current_room = new_room
      print(f"MOVED to {new_room}.")
    elif action == "CELEBRATE":
      print("Treasure found")

In [5]:
agent = ReflexAgent()
for step in range(10):
  print(f"\n --- step {step+1} --- ")
  perception =agent.percieve(WORLD)
  action= agent.think(perception)
  print(f"action: {action}")
  agent.act(action)
  if action == "CELEBRATE":
    break


Reflex agent starts in {self.current_room}.

 --- step 1 --- 
action: MOVE_TO_B
MOVED to B.

 --- step 2 --- 
action: MOVE_TO_A
MOVED to A.

 --- step 3 --- 
action: MOVE_TO_B
MOVED to B.

 --- step 4 --- 
action: MOVE_TO_A
MOVED to A.

 --- step 5 --- 
action: MOVE_TO_B
MOVED to B.

 --- step 6 --- 
action: MOVE_TO_A
MOVED to A.

 --- step 7 --- 
action: MOVE_TO_B
MOVED to B.

 --- step 8 --- 
action: MOVE_TO_A
MOVED to A.

 --- step 9 --- 
action: MOVE_TO_B
MOVED to B.

 --- step 10 --- 
action: MOVE_TO_A
MOVED to A.


# GOAL BASED AGENT


In [9]:
# A map of rooms. Each room has a list of neighboring rooms.
WORLD = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'G'],   # G has the treasure
    'F': ['C'],
    'G': ['E']
}

START_ROOM = 'A'
GOAL_ROOM = 'G'

In [10]:
class GoalBasedAgent:
    def __init__(self):
        self.current = START_ROOM
        self.visited = [START_ROOM]
        self.stack = [START_ROOM]   # path stack

    def perceive(self, world):
        return self.current, world[self.current]

    def think(self, perception, goal):
        room, neighbors = perception
        print(f"In {room}. Visited: {self.visited}, Neighbors: {neighbors}")
        if room == goal:
            return "CELEBRATE"
        for nb in neighbors:
            if nb not in self.visited:
                return f"MOVE_TO_{nb}"
        # Backtrack: pop the stack to go to previous room
        if len(self.stack) > 1:
            self.stack.pop()           # remove current dead-end
            prev = self.stack[-1]      # new top is the previous room
            return f"MOVE_TO_{prev}"
        return "STUCK"

    def act(self, action):
        if action.startswith("MOVE_TO_"):
            new_room = action.split("_")[-1]
            if new_room not in self.visited:
                self.visited.append(new_room)
                self.stack.append(new_room)   # going forward, push
            # If backtracking, stack already popped in think()
            self.current = new_room
            print(f"Moved to {new_room}. Stack: {self.stack}")
        elif action == "CELEBRATE":
            print(f"🎉 Treasure found in {self.current}! Full path: {self.stack}")
        elif action == "STUCK":
            print("Nowhere to go. Stuck!")

In [11]:
agent = GoalBasedAgent()
for step in range(15):
    print(f"\n--- Step {step+1} ---")
    perception = agent.perceive(WORLD)
    action = agent.think(perception, END_ROOM)
    print(f"Decision: {action}")
    agent.act(action)
    if action in ("CELEBRATE", "STUCK"):
        break


--- Step 1 ---
In A. Visited: ['A'], Neighbors: ['B', 'C']
Decision: MOVE_TO_B
Moved to B. Stack: ['A', 'B']

--- Step 2 ---
In B. Visited: ['A', 'B'], Neighbors: ['A', 'D', 'E']
Decision: MOVE_TO_D
Moved to D. Stack: ['A', 'B', 'D']

--- Step 3 ---
In D. Visited: ['A', 'B', 'D'], Neighbors: ['B']
Decision: MOVE_TO_B
Moved to B. Stack: ['A', 'B']

--- Step 4 ---
In B. Visited: ['A', 'B', 'D'], Neighbors: ['A', 'D', 'E']
Decision: MOVE_TO_E
Moved to E. Stack: ['A', 'B', 'E']

--- Step 5 ---
In E. Visited: ['A', 'B', 'D', 'E'], Neighbors: ['B', 'G']
Decision: MOVE_TO_G
Moved to G. Stack: ['A', 'B', 'E', 'G']

--- Step 6 ---
In G. Visited: ['A', 'B', 'D', 'E', 'G'], Neighbors: ['E']
Decision: MOVE_TO_E
Moved to E. Stack: ['A', 'B', 'E']

--- Step 7 ---
In E. Visited: ['A', 'B', 'D', 'E', 'G'], Neighbors: ['B', 'G']
Decision: MOVE_TO_B
Moved to B. Stack: ['A', 'B']

--- Step 8 ---
In B. Visited: ['A', 'B', 'D', 'E', 'G'], Neighbors: ['A', 'D', 'E']
Decision: MOVE_TO_A
Moved to A. Stack: [

In [12]:
import random

class QLearningAgent:
    def __init__(self, start, goal, world, learning_rate=0.1, discount=0.9, epsilon=0.2):
        self.current = start
        self.start = start
        self.goal = goal
        self.world = world
        self.lr = learning_rate
        self.discount = discount
        self.epsilon = epsilon

        # Q-table: dict of dicts. q_table[room][neighbor] = value
        self.q_table = {}
        for room in world:
            self.q_table[room] = {neighbor: 0.0 for neighbor in world[room]}

    def choose_action(self, room):
        # Epsilon-greedy
        if random.random() < self.epsilon:
            return random.choice(self.world[room])
        else:
            # Pick neighbor with highest Q-value
            return max(self.q_table[room], key=self.q_table[room].get)

    def act(self, action):
        self.current = action

    def learn(self, old_room, action, reward, new_room):
        old_value = self.q_table[old_room][action]
        future_best = max(self.q_table[new_room].values()) if new_room != self.goal else 0
        new_value = old_value + self.lr * (reward + self.discount * future_best - old_value)
        self.q_table[old_room][action] = new_value

# Training loop
agent = QLearningAgent(START_ROOM, GOAL_ROOM, WORLD)

for episode in range(200):
    agent.current = START_ROOM
    steps = 0
    while agent.current != GOAL_ROOM and steps < 20:
        old_room = agent.current
        action = agent.choose_action(old_room)
        agent.act(action)
        new_room = agent.current

        # Reward: +1 for reaching goal, -0.1 for each step
        reward = 1 if new_room == GOAL_ROOM else -0.1
        agent.learn(old_room, action, reward, new_room)
        steps += 1

# After training, see the learned policy
print("Learned Q-values (best direction from each room):")
for room in WORLD:
    best = max(agent.q_table[room], key=agent.q_table[room].get)
    print(f"  {room} -> {best}   (Q: {agent.q_table[room][best]:.3f})")

Learned Q-values (best direction from each room):
  A -> B   (Q: 0.620)
  B -> E   (Q: 0.800)
  C -> A   (Q: 0.391)
  D -> B   (Q: 0.538)
  E -> G   (Q: 1.000)
  F -> C   (Q: -0.002)
  G -> E   (Q: 0.000)
